# Phase 2 · Alignment fine-tuning (DDP)
Trains the student (Qwen3-Embedding-0.6B, frozen) + an adapter (LoRA r=16 on every decoder linear, linear head 1024→256) so its query vectors land on the teacher's cached query vectors (`emb/teacher/align.npy`). One process per GPU; only the adapter trains.

Checkpoints are **adapter-only** (LoRA + head weights, a few MB, plus `adapter_config.json` naming the base model): `runs/<mode>/align/step_<n>/` (resumable, last `keep_last` kept) and `runs/<mode>/align/final/`. Any of them loads with `load_encoder(dir)` / `evaluate_model(dir, ...)`.

Needs the teacher cache from `phase_1_teacher` (same `DEBUG` setting).

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import torch
from teacher import TEACHER_DEFAULTS as T

DEBUG = True    # aligns on the debug cache built by phase_1_teacher with DEBUG = True. Set False for the full run.
NPROCS = 2      # one process per T4

# --- paths: teacher cache in, adapter checkpoints out; final metrics go next to the phase_1_eval rows
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
DATA_PATH = f"{ROOT}/cache/{MODE}/data.json"
EMB_DIR = f"{ROOT}/cache/{MODE}/emb/teacher"
OUT_DIR = f"{ROOT}/runs/{MODE}/align"
EVAL_DIR = f"{ROOT}/runs/{MODE}/eval"

# --- alignment (align.ALIGN_DEFAULTS; unknown keys raise)
ALIGN = dict(model="Qwen/Qwen3-Embedding-0.6B",   # student base, frozen
             dim=T["dim"],                        # head output = teacher index dim
             lora_rank=16, lora_alpha=32, lora_dropout=0.05,
             lora_targets=("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"),
             query_prefix="", max_query_len=32,   # student queries: no prompt
             # loss: a name in align.LOSSES + its kwargs. The workers are spawned processes and can't receive a
             # function defined in a notebook cell, so a new loss goes in align.py (+ LOSSES) and is picked here.
             loss="l2", loss_kwargs={},
             lr=2e-4, head_lr=1e-3, weight_decay=0.0, warmup_ratio=0.05,
             epochs=1, batch_size=128,            # per GPU; global batch = batch_size * NPROCS
             grad_clip=1.0, amp=True, seed=0,
             log_every=20, eval_every=200,        # eval = val loss/cos + val retrieval from the teacher corpus
             save_every=500, keep_last=2,
             data_path=DATA_PATH, teacher_emb_dir=EMB_DIR, out_dir=OUT_DIR)
if DEBUG:
    ALIGN.update(log_every=5, eval_every=20, save_every=20)

## 1 · Check inputs
Teacher vectors for the alignment queries, val queries and corpus: finite, unit norm, ids in `data.json` order.

In [ ]:
from align import LOSSES
from data import load_emb, load_json
from evaluation import check_emb

assert ALIGN["loss"] in LOSSES, f"loss {ALIGN['loss']!r} not in align.LOSSES {sorted(LOSSES)}"
data = load_json(DATA_PATH)
for name in ("align", "val", "corpus"):
    emb, ids = load_emb(EMB_DIR, name)
    check_emb(emb, ids, data[name]["ids"], name)
    print(f"{name:<7} {emb.shape}")
per_step = ALIGN["batch_size"] * NPROCS
steps = len(data["align"]["ids"]) // per_step * ALIGN["epochs"]
print(f"global batch {per_step}, {steps} steps")

## 2 · Train
Resumable: re-running after a crash continues from the latest `step_<n>/`. A no-op once `final/` exists; delete `OUT_DIR` (or point it elsewhere) to train again.

In [ ]:
from align import align_main
from ddp import launch

launch(align_main, ALIGN, nprocs=NPROCS)

## 3 · Training log
From `OUT_DIR/log.jsonl`: train loss every `log_every` steps, val metrics every `eval_every`.

In [ ]:
from align import read_log

log = read_log(OUT_DIR)
train = [r for r in log if "train_loss" in r]
print("train loss:", " ".join(f"{r['step']}:{r['train_loss']:.4f}" for r in train[::max(1, len(train) // 20)]))
for r in log:
    if "val_loss" in r:
        print(f"step {r['step']:>6}  val_loss {r['val_loss']:.4f}  val_cos {r['val_cos']:.4f}  "
              f"ndcg@10 {r['val_ndcg@10']:.4f}  recall@10 {r['val_recall@10']:.4f}")

## 4 · Evaluate the final adapter
Same scoring as `phase_1_eval`, merged into its `metrics.json`. Retention needs the `teacher` row, i.e. `phase_1_eval` run on the same cache.

In [ ]:
from evaluation import evaluate_model, load_index, save_metrics, summarize
from model import get_device

device = get_device()
docs, _ = load_index(data, EMB_DIR, ("val", "test"))
name = f"aligned_r{ALIGN['lora_rank']}_{ALIGN['loss']}"
result = evaluate_model(f"{OUT_DIR}/final", data, docs, splits=("val", "test"), dim=ALIGN["dim"],
                        max_query_len=ALIGN["max_query_len"], query_prefix=ALIGN["query_prefix"], k=100,
                        device=device, dtype="fp16")
save_metrics(EVAL_DIR, {name: result})
for split in ("val", "test"):
    summarize(EVAL_DIR, split)
    print()